### Capacitated Vehicle Routing Problem (CVRP)

This variant considers that vehicles have a capacity $Q$ and that each customer $j$ has a demand $d_j$. The formulation is as follows:

$$
\begin{aligned}
\min \quad & \sum_{k \in K} \sum_{i \in N} \sum_{\substack{j \in N \\ j \ne i}} c_{ij} \cdot x_{ijk} \\
\text{s.t.} \quad
& \sum_{k \in K} \sum_{\substack{i \in N \\ i \ne j}} x_{ijk} = 1 && \forall j \in C \\
& \sum_{\substack{i \in N \\ i \ne h}} x_{ihk} = \sum_{\substack{j \in N \\ j \ne h}} x_{hjk} && \forall h \in N,\; \forall k \in K \\
& u_{ik} + d_j \cdot x_{ijk} \leq u_{jk} + Q \cdot (1 - x_{ijk}) && \forall i \ne j,\; i,j \in C,\; \forall k \in K \\
& u_{jk} \geq d_j,\quad u_{jk} \leq Q && \forall j \in C,\; \forall k \in K \\
& \sum_{j \in C} x_{0jk} = 1 && \forall k \in K \\
& \sum_{j \in C} x_{j0k} = 1 && \forall k \in K \\
& x_{ijk} \in \{0,1\} && \forall i,j \in N,\; i \ne j,\; \forall k \in K \\
& u_{ik} \in \mathbb{R} && \forall i \in N,\; \forall k \in K
\end{aligned}
$$


In [ ]:
%pip install -q amplpy numpy matplotlib pandas networkx folium
from amplpy import AMPL, ampl_notebook
import numpy as np

# HiGHS is the default. Gurobi requires an AMPL-compatible license.
SOLVER = "highs"  # or "gurobi"
LICENSE_UUID = "default"  # Colab Community Edition; use your UUID locally
runtime = ampl_notebook(modules=[SOLVER], license_uuid=LICENSE_UUID)

def new_ampl():
    return AMPL()

def solve_checked(model):
    model.solve(solver=SOLVER)
    if model.solve_result != "solved":
        raise RuntimeError(f"No proven optimal solution: {model.solve_result}. "
                           "Inspect the solver log before extracting values.")

def values(model, name):
    # Numeric dictionaries keep plotting independent of the solver API.
    return model.var[name].get_values().to_dict()

import itertools
import matplotlib.pyplot as plt
import math


In [ ]:
# Coordinates: 0 is the depot
locations = {
    0: (50, 50),  # depot
    1: (20, 30),
    2: (60, 20),
    3: (30, 70),
    4: (80, 40),
    5: (60, 80),
    6: (40, 10)
}

# Demand by customer (node 0 has demand 0)
demand = {0: 0, 1: 10, 2: 15, 3: 10, 4: 20, 5: 25, 6: 10}
Q = 50  # capacity of each vehicle
vehicles = [1, 2]
nodes = list(locations.keys())
customers = [i for i in nodes if i != 0]

# Euclidean distances
dist = {
    (i, j): round(((locations[i][0] - locations[j][0]) ** 2 +
                   (locations[i][1] - locations[j][1]) ** 2) ** 0.5)
    for i, j in itertools.permutations(nodes, 2)
}


In [ ]:
m = new_ampl()
m.eval(r"""
set N;
set C within N;
set K;
set A within N cross N;
param c {A} >= 0;
var x {A,K} binary;
minimize Total_Cost: sum {(i,j) in A,k in K} c[i,j]*x[i,j,k];
subject to Visit {j in C}: sum {(i,j) in A,k in K} x[i,j,k] = 1;
subject to Flow {h in N,k in K}:
    sum {(i,h) in A} x[i,h,k] = sum {(h,j) in A} x[h,j,k];
param demand {C} > 0;
param Q > 0;
var u {j in C,k in K} >= demand[j] <= Q;
subject to Departure {k in K}: sum {(i,j) in A: i=0} x[i,j,k] <= 1;
subject to Return {k in K}: sum {(i,j) in A: j=0} x[i,j,k] <= 1;
subject to Load {(i,j) in A,k in K: i in C and j in C}:
    u[i,k]-u[j,k] + demand[j]*x[i,j,k] <= Q*(1-x[i,j,k]);
""")
nodes = list(locations.keys())
m.set["N"] = list(nodes)
m.set["C"] = list(customers)
m.set["K"] = list(vehicles)
m.set["A"] = list(dist)
m.param["c"] = dist
m.param["demand"] = {j: demand[j] for j in customers}
m.param["Q"] = Q

solve_checked(m)
x = values(m, "x")
u = values(m, "u")


In [ ]:
# Retrieve the solutions
routes = {k: [] for k in vehicles}
vals = x

# Reconstruct routes, including unused vehicles, with bounded traversal.
for k in vehicles:
    route = [0]
    current = 0
    seen = set()
    for _ in range(len(nodes)+1):
        outgoing = [j for j in nodes if j!=current and vals.get((current,j,k),0)>0.5]
        if not outgoing:
            if current==0 and len(route)==1:
                break  # Unused vehicle.
            raise RuntimeError(f"Broken route for vehicle {k}")
        if len(outgoing)!=1:
            raise RuntimeError(f"Ambiguous successor for vehicle {k}")
        current = outgoing[0]
        route.append(current)
        if current==0:
            break
        if current in seen:
            raise RuntimeError(f"Subtour for vehicle {k}")
        seen.add(current)
    else:
        raise RuntimeError("Route traversal exceeded the number of nodes")
    routes[k] = route

visited = [j for route in routes.values() for j in route if j!=0]
assert sorted(visited)==sorted(customers), "Customers missing or duplicated"
# Function to plot the routes
def plot_routes(locations, routes, title="Routes by Vehicle"):
    colors = ['red', 'blue', 'green', 'orange', 'purple']
    plt.figure(figsize=(6, 6))
    for i in locations:
        x, y = locations[i]
        plt.plot(x, y, 'ko')
        plt.text(x + 1, y + 1, str(i), fontsize=12)

    for k, route in routes.items():
        for i in range(len(route) - 1):
            a, b = route[i], route[i + 1]
            x_coords = [locations[a][0], locations[b][0]]
            y_coords = [locations[a][1], locations[b][1]]
            plt.plot(
                x_coords, y_coords, '-',
                color=colors[k % len(colors)],
                label=f'Vehicle {k}' if i == 0 else ""
            )

    plt.title(title)
    plt.xlabel("X")
    plt.ylabel("Y")
    plt.grid(True)
    plt.legend()
    plt.show()

# Plot
plot_routes(locations, routes)


In [ ]:
import random
import itertools

def generate_cvrp_instance(num_customers=15, num_vehicles=3, vehicle_capacity=35, seed=42):
    random.seed(seed)

    # Create nodes (0 = depot)
    locations = {0: (random.randint(40, 60), random.randint(40, 60))}  # centered depot
    for i in range(1, num_customers + 1):
        locations[i] = (random.randint(0, 100), random.randint(0, 100))

    # Create random demands (customers only)
    demand = {0: 0}
    total_demand = 0
    for j in range(1, num_customers + 1):
        d = random.randint(5, 7)
        demand[j] = d
        total_demand += d

    # Ensure that total vehicle capacity is sufficient
    total_capacity = num_vehicles * vehicle_capacity
    assert total_capacity >= total_demand, (
        f"Insufficient capacity: total demand = {total_demand}, available capacity = {total_capacity}"
    )

    # Compute Euclidean distances
    nodes = list(locations.keys())
    # dist = {
    #     (i, j): round(((locations[i][0] - locations[j][0]) ** 2 +
    #                    (locations[i][1] - locations[j][1]) ** 2) ** 0.5)
    #     for i, j in itertools.permutations(nodes, 2)
    # }

    dist = {
        (i, j): math.floor(((locations[i][0] - locations[j][0]) ** 2 +
                       (locations[i][1] - locations[j][1]) ** 2) ** 0.5)
        for i, j in itertools.permutations(nodes, 2)
    }

    # Customers and vehicles
    customers = [i for i in nodes if i != 0]
    vehicles = list(range(1, num_vehicles + 1))

    return locations, demand, dist, customers, vehicles, vehicle_capacity

In [ ]:
locations, demand, dist, customers, vehicles, Q = generate_cvrp_instance(seed= 42)

In [ ]:
m = new_ampl()
m.eval(r"""
set N;
set C within N;
set K;
set A within N cross N;
param c {A} >= 0;
var x {A,K} binary;
minimize Total_Cost: sum {(i,j) in A,k in K} c[i,j]*x[i,j,k];
subject to Visit {j in C}: sum {(i,j) in A,k in K} x[i,j,k] = 1;
subject to Flow {h in N,k in K}:
    sum {(i,h) in A} x[i,h,k] = sum {(h,j) in A} x[h,j,k];
param demand {C} > 0;
param Q > 0;
var u {j in C,k in K} >= demand[j] <= Q;
subject to Departure {k in K}: sum {(i,j) in A: i=0} x[i,j,k] <= 1;
subject to Return {k in K}: sum {(i,j) in A: j=0} x[i,j,k] <= 1;
subject to Load {(i,j) in A,k in K: i in C and j in C}:
    u[i,k]-u[j,k] + demand[j]*x[i,j,k] <= Q*(1-x[i,j,k]);
""")
nodes = list(locations.keys())
m.set["N"] = list(nodes)
m.set["C"] = list(customers)
m.set["K"] = list(vehicles)
m.set["A"] = list(dist)
m.param["c"] = dist
m.param["demand"] = {j: demand[j] for j in customers}
m.param["Q"] = Q

solve_checked(m)
x = values(m, "x")
u = values(m, "u")


In [ ]:
# Retrieve the solutions
routes = {k: [] for k in vehicles}
vals = x

# Reconstruct routes, including unused vehicles, with bounded traversal.
for k in vehicles:
    route = [0]
    current = 0
    seen = set()
    for _ in range(len(nodes)+1):
        outgoing = [j for j in nodes if j!=current and vals.get((current,j,k),0)>0.5]
        if not outgoing:
            if current==0 and len(route)==1:
                break  # Unused vehicle.
            raise RuntimeError(f"Broken route for vehicle {k}")
        if len(outgoing)!=1:
            raise RuntimeError(f"Ambiguous successor for vehicle {k}")
        current = outgoing[0]
        route.append(current)
        if current==0:
            break
        if current in seen:
            raise RuntimeError(f"Subtour for vehicle {k}")
        seen.add(current)
    else:
        raise RuntimeError("Route traversal exceeded the number of nodes")
    routes[k] = route

visited = [j for route in routes.values() for j in route if j!=0]
assert sorted(visited)==sorted(customers), "Customers missing or duplicated"
# Function to plot the routes
def plot_routes(locations, routes, title="Routes by Vehicle"):
    colors = ['red', 'blue', 'green', 'orange', 'purple']
    plt.figure(figsize=(6, 6))
    for i in locations:
        x, y = locations[i]
        plt.plot(x, y, 'ko')
        plt.text(x + 1, y + 1, str(i), fontsize=12)

    for k, route in routes.items():
        for i in range(len(route) - 1):
            a, b = route[i], route[i + 1]
            x_coords = [locations[a][0], locations[b][0]]
            y_coords = [locations[a][1], locations[b][1]]
            plt.plot(
                x_coords, y_coords, '-',
                color=colors[k % len(colors)],
                label=f'Vehicle {k}' if i == 0 else ""
            )

    plt.title(title)
    plt.xlabel("X")
    plt.ylabel("Y")
    plt.grid(True)
    plt.legend()
    plt.show()

# Plot
plot_routes(locations, routes)